In [1]:
import numpy as np

**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [2]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # The easiest case:

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [3]:
class Sequential(Module):
    """
         This class implements a container, which processes `input` data sequentially.

         `input` is processed by each module (layer) in self.modules consecutively.
         The resulting array is called `output`.
    """

    def __init__ (self):
        super(Sequential, self).__init__()
        self.modules = []

    def add(self, module):
        """
        Adds a module to the container.
        """
        self.modules.append(module)

    def updateOutput(self, input):
        """
        Basic workflow of FORWARD PASS:

            y_0    = module[0].forward(input)
            y_1    = module[1].forward(y_0)
            ...
            output = module[n-1].forward(y_{n-2})


        Just write a little loop.
        """

        self._inputs = []
        x = input
        for m in self.modules:
            self._inputs.append(x)
            x = m.forward(x)
        self.output = x
        return self.output

    def backward(self, input, gradOutput):
        """
        Workflow of BACKWARD PASS:

            g_{n-1} = module[n-1].backward(y_{n-2}, gradOutput)
            g_{n-2} = module[n-2].backward(y_{n-3}, g_{n-1})
            ...
            g_1 = module[1].backward(y_0, g_2)
            gradInput = module[0].backward(input, g_1)


        !!!

        To ech module you need to provide the input, module saw while forward pass,
        it is used while computing gradients.
        Make sure that the input for `i-th` layer the output of `module[i]` (just the same input as in forward pass)
        and NOT `input` to this Sequential module.

        !!!

        """
        grad = gradOutput
        for i in range(len(self.modules) - 1, -1, -1):
            grad = self.modules[i].backward(self._inputs[i], grad)
        self.gradInput = grad
        return self.gradInput


    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        """
        Should gather all parameters in a list.
        """
        return [x.getParameters() for x in self.modules]

    def getGradParameters(self):
        """
        Should gather all gradients w.r.t parameters in a list.
        """
        return [x.getGradParameters() for x in self.modules]

    def __repr__(self):
        string = "".join([str(x) + '\n' for x in self.modules])
        return string

    def __getitem__(self,x):
        return self.modules.__getitem__(x)

    def train(self):
        """
        Propagates training parameter through all modules
        """
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        """
        Propagates training parameter through all modules
        """
        self.training = False
        for module in self.modules:
            module.evaluate()



# Layers

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [4]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # This is a nice initialization
        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size = n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = input @ self.W.T + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput @ self.W
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradW = gradOutput.T @ input
        self.gradb = np.sum(gradOutput, axis=0)
        pass

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q

## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [5]:
class SoftMax(Module):
    def __init__(self):
         super(SoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))

        # Your code goes here. ################################################
        exp_x = np.exp(self.output)
        self.output = exp_x / np.sum(exp_x, axis=1, keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        s = self.output
        self.gradInput = s * (gradOutput - np.sum(gradOutput * s, axis=1, keepdims=True))
        return self.gradInput

    def __repr__(self):
        return "SoftMax"

## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [6]:
class LogSoftMax(Module):
    def __init__(self):
         super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))

        # Your code goes here. ################################################
        logsum = np.log(np.sum(np.exp(self.output), axis=1, keepdims=True))
        self.output = self.output - logsum
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        exp_x = np.exp(np.subtract(input, input.max(axis=1, keepdims=True)))
        softmax = exp_x / np.sum(exp_x, axis=1, keepdims=True)
        self.gradInput = gradOutput - softmax * np.sum(gradOutput, axis=1, keepdims=True)
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"

## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [7]:
class BatchNormalization(Module):
    EPS = 1e-3
    def __init__(self, alpha = 0.):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha
        self.moving_mean = None
        self.moving_variance = None

    def updateOutput(self, input):
        # Your code goes here. ################################################
        # use self.EPS please
        n_feats = input.shape[1]
        if self.moving_mean is None:
            self.moving_mean = np.zeros(n_feats, dtype=input.dtype)
            self.moving_variance = np.ones(n_feats, dtype=input.dtype)

        if self.training:
            mu = np.mean(input, axis=0)
            var = np.var(input, axis=0)
            self.moving_mean = self.moving_mean * self.alpha + mu * (1.0 - self.alpha)
            self.moving_variance = self.moving_variance * self.alpha + var * (1.0 - self.alpha)
            std = np.sqrt(var + self.EPS)
            self._bn_mu = mu
            self._bn_var = var
            self._bn_std = std
            self._bn_xhat = (input - mu) / std
            self.output = self._bn_xhat
        else:
            std = np.sqrt(self.moving_variance + self.EPS)
            self.output = (input - self.moving_mean) / std
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if not self.training:
            std = np.sqrt(self.moving_variance + self.EPS)
            self.gradInput = gradOutput / std
            return self.gradInput

        mu = self._bn_mu
        std = self._bn_std
        x_hat = self._bn_xhat
        dxhat = gradOutput
        self.gradInput = (1.0 / std) * (
            dxhat - np.mean(dxhat, axis=0) - x_hat * np.mean(dxhat * x_hat, axis=0)
        )
        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"

In [8]:
class ChannelwiseScaling(Module):
    """
       Implements linear transform of input y = \gamma * x + \beta
       where \gamma, \beta - learnable vectors of length x.shape[-1]
    """
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta = np.sum(gradOutput, axis=0)
        self.gradGamma = np.sum(gradOutput*input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [9]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()

        self.p = p
        self.mask = None

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if not self.training:
            self.output = input.copy()
            return self.output
        if self.p >= 1.0:
            self.mask = np.zeros_like(input)
            self.output = np.zeros_like(input)
            return self.output
        if self.p <= 0.0:
            self.mask = np.ones_like(input)
            self.output = input.copy()
            return self.output
        self.mask = (np.random.rand(*input.shape) >= self.p).astype(input.dtype)
        scale = 1.0 / (1.0 - self.p)
        self.output = input * self.mask * scale
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if not self.training:
            self.gradInput = gradOutput.copy()
            return self.gradInput
        if self.p >= 1.0:
            self.gradInput = np.zeros_like(gradOutput)
            return self.gradInput
        if self.p <= 0.0:
            self.gradInput = gradOutput.copy()
            return self.gradInput
        scale = 1.0 / (1.0 - self.p)
        self.gradInput = gradOutput * self.mask * scale
        return self.gradInput

    def __repr__(self):
        return "Dropout"

#6. (2.0) Conv2d
Implement [**Conv2d**](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use only this list of parameters: (in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode) and fix dilation=1 and groups=1.

In [10]:
def _conv_pair(x):
    if isinstance(x, (tuple, list)):
        return int(x[0]), int(x[1])
    return int(x), int(x)


def _conv_same_pad_hw(H, W, kH, kW, sH, sW, dH=1, dW=1):
    out_h = (H + sH - 1) // sH
    out_w = (W + sW - 1) // sW
    pad_h = max((out_h - 1) * sH + (kH - 1) * dH - H + 1, 0)
    pad_w = max((out_w - 1) * sW + (kW - 1) * dW - W + 1, 0)
    return pad_h, pad_w


def _conv_resolve_padding(padding, H_in, W_in, kH, kW, sH, sW):
    if padding == "same":
        pad_h, pad_w = _conv_same_pad_hw(H_in, W_in, kH, kW, sH, sW)
        pad_t = pad_h // 2
        pad_b = pad_h - pad_t
        pad_l = pad_w // 2
        pad_r = pad_w - pad_l
        return pad_t, pad_b, pad_l, pad_r
    if isinstance(padding, int):
        p = padding
        return p, p, p, p
    ph, pw = _conv_pair(padding)
    return ph, ph, pw, pw


def _conv_pad2d(x, pad_t, pad_b, pad_l, pad_r, mode):
    if mode == "zeros":
        return np.pad(
            x,
            ((0, 0), (0, 0), (pad_t, pad_b), (pad_l, pad_r)),
            mode="constant",
            constant_values=0,
        )
    if mode == "replicate":
        return np.pad(x, ((0, 0), (0, 0), (pad_t, pad_b), (pad_l, pad_r)), mode="edge")
    if mode == "reflect":
        return np.pad(x, ((0, 0), (0, 0), (pad_t, pad_b), (pad_l, pad_r)), mode="reflect")
    raise ValueError(mode)


def _conv_replicate_pad_backward(g, pad_t, pad_b, pad_l, pad_r, H, W):
    N, C, Hp, Wp = g.shape
    grad = np.zeros((N, C, H, W), dtype=g.dtype)
    ii, jj = np.meshgrid(np.arange(Hp), np.arange(Wp), indexing="ij")
    si = np.clip(ii - pad_t, 0, H - 1)
    sj = np.clip(jj - pad_l, 0, W - 1)
    for n in range(N):
        for c in range(C):
            for pi in range(Hp):
                for pj in range(Wp):
                    grad[n, c, si[pi, pj], sj[pi, pj]] += g[n, c, pi, pj]
    return grad


def _conv_reflect_pad_backward(grad_padded, pad_t, pad_b, pad_l, pad_r, H, W):
    N, C, Hp, Wp = grad_padded.shape
    grad = np.zeros((N, C, H, W), dtype=grad_padded.dtype)
    tmp = np.arange(H * W, dtype=np.int64).reshape(H, W)
    tmp_p = np.pad(tmp, ((pad_t, pad_b), (pad_l, pad_r)), mode="reflect")
    oi = (tmp_p // W).astype(np.int64)
    oj = (tmp_p % W).astype(np.int64)
    for i in range(Hp):
        for j in range(Wp):
            si, sj = int(oi[i, j]), int(oj[i, j])
            grad[:, :, si, sj] += grad_padded[:, :, i, j]
    return grad


def _conv_pad_backward(grad_padded, original_shape, pad_t, pad_b, pad_l, pad_r, mode):
    N, C, H, W = original_shape
    g = grad_padded
    if mode == "zeros":
        return g[:, :, pad_t : pad_t + H, pad_l : pad_l + W].copy()
    if mode == "replicate":
        return _conv_replicate_pad_backward(g, pad_t, pad_b, pad_l, pad_r, H, W)
    if mode == "reflect":
        return _conv_reflect_pad_backward(grad_padded, pad_t, pad_b, pad_l, pad_r, H, W)
    raise ValueError(mode)


def _conv_im2col(x, kH, kW, sH, sW):
    N, C, H, W = x.shape
    OH = (H - kH) // sH + 1
    OW = (W - kW) // sW + 1
    cols = np.zeros((N, C * kH * kW, OH * OW), dtype=x.dtype)
    for n in range(N):
        col_idx = 0
        for oh in range(OH):
            for ow in range(OW):
                patch = x[n, :, oh * sH : oh * sH + kH, ow * sW : ow * sW + kW]
                cols[n, :, col_idx] = patch.ravel()
                col_idx += 1
    return cols, OH, OW


def _conv_col2im(cols, N, C, H, W, kH, kW, sH, sW, OH, OW):
    grad = np.zeros((N, C, H, W), dtype=cols.dtype)
    for n in range(N):
        col_idx = 0
        for oh in range(OH):
            for ow in range(OW):
                patch = cols[n, :, col_idx].reshape(C, kH, kW)
                grad[n, :, oh * sH : oh * sH + kH, ow * sW : ow * sW + kW] += patch
                col_idx += 1
    return grad


def _conv2d_forward(input, weight, bias, stride, padding, padding_mode):
    N, C_in, H_in, W_in = input.shape
    C_out, _, kH, kW = weight.shape
    sH, sW = _conv_pair(stride)
    pad_t, pad_b, pad_l, pad_r = _conv_resolve_padding(padding, H_in, W_in, kH, kW, sH, sW)
    x_pad = _conv_pad2d(input, pad_t, pad_b, pad_l, pad_r, padding_mode)
    cols, OH, OW = _conv_im2col(x_pad, kH, kW, sH, sW)
    W_mat = weight.reshape(C_out, -1)
    out_flat = np.einsum("oc,ncp->nop", W_mat, cols)
    out = out_flat.reshape(N, C_out, OH, OW)
    if bias is not None:
        out = out + bias.reshape(1, -1, 1, 1)
    pad_info = (pad_t, pad_b, pad_l, pad_r, x_pad.shape, OH, OW)
    return out, pad_info


def _conv2d_grad_input(grad_out, weight, stride, pad_info, padding_mode, orig_shape):
    pad_t, pad_b, pad_l, pad_r, padded_shape, OH, OW = pad_info
    N, C_in, H_in, W_in = orig_shape
    C_out, _, kH, kW = weight.shape
    sH, sW = _conv_pair(stride)
    _, _, H_pad, W_pad = padded_shape
    W_mat = weight.reshape(C_out, -1)
    grad_flat = grad_out.reshape(N, C_out, OH * OW)
    gcols = np.einsum("co,nop->ncp", W_mat.T, grad_flat)
    grad_pad = _conv_col2im(gcols, N, C_in, H_pad, W_pad, kH, kW, sH, sW, OH, OW)
    return _conv_pad_backward(grad_pad, orig_shape, pad_t, pad_b, pad_l, pad_r, padding_mode)


class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=0, bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.bias = bias
        self.padding_mode = padding_mode

        k_h, k_w = (kernel_size, kernel_size) if isinstance(kernel_size, int) else tuple(kernel_size)
        fan_in = in_channels * k_h * k_w
        stdv = 1.0 / np.sqrt(fan_in)
        self.weight = np.random.uniform(-stdv, stdv, size=(out_channels, in_channels, k_h, k_w))
        if bias:
            self.bias = np.random.uniform(-stdv, stdv, size=out_channels)
        else:
            self.bias = None
        self.grad_weight = np.zeros_like(self.weight)
        if bias:
            self.grad_bias = np.zeros(out_channels, dtype=self.weight.dtype)
        else:
            self.grad_bias = None

    def updateOutput(self, input):
        self.output, self._pad_info = _conv2d_forward(
            input, self.weight, self.bias, self.stride, self.padding, self.padding_mode
        )
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = _conv2d_grad_input(
            gradOutput,
            self.weight,
            self.stride,
            self._pad_info,
            self.padding_mode,
            input.shape,
        )
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        pad_t, pad_b, pad_l, pad_r, padded_shape, OH, OW = self._pad_info
        N, C_in, H_in, W_in = input.shape
        C_out, _, kH, kW = self.weight.shape
        sH, sW = _conv_pair(self.stride)
        x_pad = _conv_pad2d(input, pad_t, pad_b, pad_l, pad_r, self.padding_mode)
        cols, _, _ = _conv_im2col(x_pad, kH, kW, sH, sW)
        P = OH * OW
        grad_flat = gradOutput.reshape(N, C_out, P)
        grad_mat = np.einsum("nop,ncp->oc", grad_flat, cols)
        self.grad_weight = grad_mat.reshape(self.weight.shape)
        if self.bias is not None:
            self.grad_bias = np.sum(gradOutput, axis=(0, 2, 3))

    def zeroGradParameters(self):
        self.grad_weight.fill(0)
        if self.grad_bias is not None:
            self.grad_bias.fill(0)

    def getParameters(self):
        if self.bias is not None:
            return [self.weight, self.bias]
        return [self.weight]

    def getGradParameters(self):
        if self.grad_bias is not None:
            return [self.grad_weight, self.grad_bias]
        return [self.grad_weight]

    def __repr__(self):
        return "Conv2d"

#7. (0.5) Implement [**MaxPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [**AvgPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html). Use only parameters like kernel_size, stride, padding (negative infinity for maxpool and zero for avgpool) and other parameters fixed as in framework.

In [11]:
def _pool_pair(x):
    if isinstance(x, (tuple, list)):
        return int(x[0]), int(x[1])
    return int(x), int(x)


class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(MaxPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        kH, kW = _pool_pair(self.kernel_size)
        sH, sW = _pool_pair(self.stride)
        pH, pW = _pool_pair(self.padding)
        neg_inf = -np.inf
        xp = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode="constant",
            constant_values=neg_inf,
        )
        N, C, Hp, Wp = xp.shape
        OH = (Hp - kH) // sH + 1
        OW = (Wp - kW) // sW + 1
        self.output = np.zeros((N, C, OH, OW), dtype=input.dtype)
        self._max_i = np.zeros((N, C, OH, OW), dtype=np.int64)
        self._max_j = np.zeros((N, C, OH, OW), dtype=np.int64)
        for n in range(N):
            for c in range(C):
                for oh in range(OH):
                    for ow in range(OW):
                        patch = xp[n, c, oh * sH : oh * sH + kH, ow * sW : ow * sW + kW]
                        flat = patch.ravel()
                        mi = int(np.argmax(flat))
                        ih, iw = mi // kW, mi % kW
                        iah = oh * sH + ih
                        jaw = ow * sW + iw
                        self.output[n, c, oh, ow] = xp[n, c, iah, jaw]
                        self._max_i[n, c, oh, ow] = iah
                        self._max_j[n, c, oh, ow] = jaw
        self._pool_input_shape = input.shape
        self._pool_pad = (pH, pW)
        self._kH, self._kW = kH, kW
        self._sH, self._sW = sH, sW
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = input.shape
        pH, pW = self._pool_pad
        kH, kW = self._kH, self._kW
        sH, sW = self._sH, self._sW
        Hp = H + 2 * pH
        Wp = W + 2 * pW
        grad_pad = np.zeros((N, C, Hp, Wp), dtype=input.dtype)
        OH, OW = gradOutput.shape[2], gradOutput.shape[3]
        for n in range(N):
            for c in range(C):
                for oh in range(OH):
                    for ow in range(OW):
                        gi = int(self._max_i[n, c, oh, ow])
                        gj = int(self._max_j[n, c, oh, ow])
                        grad_pad[n, c, gi, gj] += gradOutput[n, c, oh, ow]
        if pH == 0 and pW == 0:
            self.gradInput = grad_pad
        else:
            self.gradInput = grad_pad[:, :, pH : pH + H, pW : pW + W]
        return self.gradInput

    def __repr__(self):
        return "MaxPool2d"

class AvgPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(AvgPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        kH, kW = _pool_pair(self.kernel_size)
        sH, sW = _pool_pair(self.stride)
        pH, pW = _pool_pair(self.padding)
        xp = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode="constant",
            constant_values=0,
        )
        N, C, Hp, Wp = xp.shape
        OH = (Hp - kH) // sH + 1
        OW = (Wp - kW) // sW + 1
        self.output = np.zeros((N, C, OH, OW), dtype=input.dtype)
        self._scale = 1.0 / (kH * kW)
        for n in range(N):
            for c in range(C):
                for oh in range(OH):
                    for ow in range(OW):
                        patch = xp[n, c, oh * sH : oh * sH + kH, ow * sW : ow * sW + kW]
                        self.output[n, c, oh, ow] = np.mean(patch)
        self._pool_input_shape = input.shape
        self._pool_pad = (pH, pW)
        self._kH, self._kW = kH, kW
        self._sH, self._sW = sH, sW
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = input.shape
        pH, pW = self._pool_pad
        kH, kW = self._kH, self._kW
        sH, sW = self._sH, self._sW
        Hp = H + 2 * pH
        Wp = W + 2 * pW
        grad_pad = np.zeros((N, C, Hp, Wp), dtype=input.dtype)
        OH, OW = gradOutput.shape[2], gradOutput.shape[3]
        scale = 1.0 / (kH * kW)
        for n in range(N):
            for c in range(C):
                for oh in range(OH):
                    for ow in range(OW):
                        g = gradOutput[n, c, oh, ow] * scale
                        grad_pad[n, c, oh * sH : oh * sH + kH, ow * sW : ow * sW + kW] += g
        if pH == 0 and pW == 0:
            self.gradInput = grad_pad
        else:
            self.gradInput = grad_pad[:, :, pH : pH + H, pW : pW + W]
        return self.gradInput

    def __repr__(self):
        return "AvgPool2d"

#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

In [12]:
class GlobalMaxPool2d(Module):
    def __init__(self):
        super(GlobalMaxPool2d, self).__init__()

    def updateOutput(self, input):
        # input: N x C x H x W -> N x C x 1 x 1
        self._in_shape = input.shape
        self._flat = input.reshape(input.shape[0], input.shape[1], -1)
        idx = np.argmax(self._flat, axis=2)
        self._max_idx = idx
        self.output = np.max(self._flat, axis=2, keepdims=True).reshape(
            input.shape[0], input.shape[1], 1, 1
        )
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = self._in_shape
        g = gradOutput.reshape(N, C)
        self.gradInput = np.zeros(self._in_shape, dtype=input.dtype)
        for n in range(N):
            for c in range(C):
                k = int(self._max_idx[n, c])
                hi, wi = divmod(k, W)
                self.gradInput[n, c, hi, wi] += g[n, c]
        return self.gradInput

    def __repr__(self):
        return "GlobalMaxPool2d"


class GlobalAvgPool2d(Module):
    def __init__(self):
        super(GlobalAvgPool2d, self).__init__()

    def updateOutput(self, input):
        self._in_shape = input.shape
        self.output = np.mean(input, axis=(2, 3), keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = self._in_shape
        scale = 1.0 / (H * W)
        self.gradInput = np.ones(self._in_shape, dtype=input.dtype) * gradOutput * scale
        return self.gradInput

    def __repr__(self):
        return "GlobalAvgPool2d"

#9. (0.2) Implement [**Flatten**](https://pytorch.org/docs/stable/generated/torch.flatten.html)

In [13]:
class Flatten(Module):
    def __init__(self, start_dim=0, end_dim=-1):
        super(Flatten, self).__init__()

        self.start_dim = start_dim
        self.end_dim = end_dim

    def updateOutput(self, input):
        # Your code goes here. ################################################
        nd = input.ndim
        s = self.start_dim
        e = self.end_dim
        if e < 0:
            e = nd + e
        if s < 0:
            s = nd + s
        self._in_shape = input.shape
        new_shape = []
        for i in range(0, s):
            new_shape.append(input.shape[i])
        prod = 1
        for i in range(s, e + 1):
            prod *= input.shape[i]
        new_shape.append(prod)
        for i in range(e + 1, nd):
            new_shape.append(input.shape[i])
        self.output = input.reshape(*new_shape)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput.reshape(self._in_shape)

        return self.gradInput

    def __repr__(self):
        return "Flatten"

# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [14]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [15]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        self.output = np.where(input >= 0, input, self.slope * input)
        return self.output

    def updateGradInput(self, input, gradOutput):
        m = (input >= 0).astype(input.dtype) + (input < 0).astype(input.dtype) * self.slope
        self.gradInput = gradOutput * m
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"

## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [16]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.alpha * (np.exp(input) - 1))
        return self.output

    def updateGradInput(self, input, gradOutput):
        g = np.where(input > 0, 1.0, self.alpha * np.exp(input))
        self.gradInput = gradOutput * g
        return self.gradInput

    def __repr__(self):
        return "ELU"

## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [17]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        self.output = np.log1p(np.exp(np.clip(input, -500, 500)))
        return self.output

    def updateGradInput(self, input, gradOutput):
        s = 1.0 / (1.0 + np.exp(-np.clip(input, -500, 500)))
        self.gradInput = gradOutput * s
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"

#13. (0.2) Gelu
Implement [**Gelu**](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html) activations.

In [18]:
class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()

    def updateOutput(self, input):
        x = input
        erf = np.vectorize(__import__("math").erf)
        self.output = 0.5 * x * (1.0 + erf(x / np.sqrt(2.0)))
        return self.output

    def updateGradInput(self, input, gradOutput):
        x = input
        erf = np.vectorize(__import__("math").erf)
        phi = np.exp(-0.5 * x * x) / np.sqrt(2.0 * np.pi)
        Phi = 0.5 * (1.0 + erf(x / np.sqrt(2.0)))
        g = Phi + x * phi
        self.gradInput = gradOutput * g
        return self.gradInput

    def __repr__(self):
        return "Gelu"

# Criterions

Criterions are used to score the models answers.

In [19]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [20]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target,2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [21]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15
    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        self.output = -np.mean(np.sum(target * np.log(input_clamp), axis=1))
        return self.output

    def updateGradInput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        n = input.shape[0]
        self.gradInput = -(target / input_clamp) / n
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"

## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [22]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = -np.mean(np.sum(target * input, axis=1))
        return self.output

    def updateGradInput(self, input, target):
        n = input.shape[0]
        self.gradInput = -target / n
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"

1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.